In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, FloatSlider, HBox, VBox, Layout, HTML, HTMLMath
from IPython.display import display

# ------------------------------------------------------------
# INTRODUCTION
# ------------------------------------------------------------

intro_html = HTML("""
<div style="font-size:14px; line-height:1.55; width:980px; padding:8px 12px; margin-bottom:10px;">
<b>Second-Order All-Pass Filter Explorer</b><br><br>
This notebook illustrates the behavior of a second-order all-pass filter.
Unlike low-pass, high-pass, band-pass, and band-stop filters, an all-pass filter
preserves the magnitude of every frequency component, so that
<b>|H(jω)| = 1</b> for all frequencies.
Its effect appears entirely in the <b>phase response</b> and the corresponding
<b>group delay</b>.
Use the sliders to vary the natural frequency <b>ω<sub>0</sub></b> and the
quality factor <b>Q</b>. Observe how these parameters change the phase and
group-delay characteristics while leaving the magnitude response unchanged.
The pole-zero diagram also shows the characteristic mirror symmetry between
the poles in the left half-plane and the zeros in the right half-plane.
</div>
""")

# ------------------------------------------------------------
# CONTROLS
# ------------------------------------------------------------

w0_title = HTML(value="<b>Natural Frequency ω<sub>0</sub> (rad/s)</b>")
w0_slider = FloatSlider(min=0.5, max=5.0, step=0.1, value=2.0, description='', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='300px'))

q_title = HTML(value="<b>Quality Factor Q</b>")
q_slider = FloatSlider(min=0.5, max=5.0, step=0.1, value=1.0, description='', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='300px'))

w0_box = VBox([w0_title, w0_slider], layout=Layout(width='320px'))
q_box = VBox([q_title, q_slider], layout=Layout(width='320px'))

control_row = HBox([w0_box, q_box], layout=Layout(width='680px', justify_content='space-between', align_items='flex-start'))

# ------------------------------------------------------------
# CURRENT PARAMETERS
# ------------------------------------------------------------

parameter_title = HTML(value="<b>Current Filter Characteristics</b>")

transfer_output = HTMLMath(layout=Layout(width='950px'))
pole_zero_output = HTMLMath(layout=Layout(width='950px'))

parameter_box = VBox([parameter_title, transfer_output, pole_zero_output], layout=Layout(width='970px', margin='8px 0 8px 0'))

# ------------------------------------------------------------
# MAIN INTERACTIVE FUNCTION
# ------------------------------------------------------------

def plot_allpass_filter(w0, Q):

    # --------------------------------------------------------
    # TRANSFER FUNCTION
    # --------------------------------------------------------

    transfer_output.value = rf"$$H(s)=\frac{{s^2-\frac{{\omega_0}}{{Q}}s+\omega_0^2}}{{s^2+\frac{{\omega_0}}{{Q}}s+\omega_0^2}}\qquad\qquad \omega_0={w0:.2f}\ \mathrm{{rad/s}},\quad Q={Q:.2f}$$"

    # --------------------------------------------------------
    # POLES AND ZEROS
    # --------------------------------------------------------

    discriminant = 1.0 / (4.0 * Q**2) - 1.0

    if discriminant >= 0.0:

        root_term = np.sqrt(discriminant)

        p1 = w0 * (-1.0 / (2.0 * Q) + root_term)
        p2 = w0 * (-1.0 / (2.0 * Q) - root_term)

        z1 = w0 * (1.0 / (2.0 * Q) + root_term)
        z2 = w0 * (1.0 / (2.0 * Q) - root_term)

    else:

        imag_term = np.sqrt(-discriminant)

        p1 = w0 * (-1.0 / (2.0 * Q) + 1j * imag_term)
        p2 = w0 * (-1.0 / (2.0 * Q) - 1j * imag_term)

        z1 = w0 * (1.0 / (2.0 * Q) + 1j * imag_term)
        z2 = w0 * (1.0 / (2.0 * Q) - 1j * imag_term)

    pole_zero_output.value = rf"$$p_{{1,2}}=\omega_0\left(-\frac{{1}}{{2Q}}\pm\sqrt{{\frac{{1}}{{4Q^2}}-1}}\right)\qquad\qquad z_{{1,2}}=\omega_0\left(\frac{{1}}{{2Q}}\pm\sqrt{{\frac{{1}}{{4Q^2}}-1}}\right)$$"

    # --------------------------------------------------------
    # FREQUENCY RANGE
    # --------------------------------------------------------

    omega = np.logspace(np.log10(w0 / 20.0), np.log10(w0 * 20.0), 2500)

    jw = 1j * omega

    # --------------------------------------------------------
    # FREQUENCY RESPONSE
    # --------------------------------------------------------

    H = (jw**2 - (w0 / Q)*jw + w0**2) / (jw**2 + (w0 / Q)*jw + w0**2)

    magnitude = np.abs(H)

    phase = np.unwrap(np.angle(H))
    phase_deg = np.degrees(phase)

    group_delay = -np.gradient(phase, omega)

    # --------------------------------------------------------
    # FIGURE
    # --------------------------------------------------------

    fig, axes = plt.subplots(2, 2, figsize=(7.8, 5.25))

    ax_mag = axes[0, 0]
    ax_phase = axes[0, 1]
    ax_group = axes[1, 0]
    ax_pz = axes[1, 1]

    # --------------------------------------------------------
    # MAGNITUDE RESPONSE
    # --------------------------------------------------------

    ax_mag.semilogx(omega, magnitude, 'r-', linewidth=2.0)
    ax_mag.axvline(w0, color='gray', linestyle='--', linewidth=1.0)

    ax_mag.set_title('Magnitude Response', fontsize=10)
    ax_mag.set_xlabel(r'Angular Frequency $\omega$ (rad/s)')
    ax_mag.set_ylabel(r'$|H(j\omega)|$')
    ax_mag.set_xlim(w0 / 20.0, w0 * 20.0)
    ax_mag.set_ylim(0.8, 1.2)

    ax_mag.axhline(1.0, color='black', linestyle=':', linewidth=1.0)
    ax_mag.grid(True, which='both', linestyle=':', alpha=0.7)

    # --------------------------------------------------------
    # PHASE RESPONSE
    # --------------------------------------------------------

    ax_phase.semilogx(omega, phase_deg, 'r-', linewidth=2.0)
    ax_phase.axvline(w0, color='gray', linestyle='--', linewidth=1.0)

    ax_phase.set_title('Phase Response', fontsize=10)
    ax_phase.set_xlabel(r'Angular Frequency $\omega$ (rad/s)')
    ax_phase.set_ylabel('Phase (deg)')
    ax_phase.set_xlim(w0 / 20.0, w0 * 20.0)

    ax_phase.grid(True, which='both', linestyle=':', alpha=0.7)

    # --------------------------------------------------------
    # GROUP DELAY
    # --------------------------------------------------------

    ax_group.semilogx(omega, group_delay, 'r-', linewidth=2.0)
    ax_group.axvline(w0, color='gray', linestyle='--', linewidth=1.0)

    ax_group.set_title('Group Delay', fontsize=10)
    ax_group.set_xlabel(r'Angular Frequency $\omega$ (rad/s)')
    ax_group.set_ylabel('Group Delay (s)')
    ax_group.set_xlim(w0 / 20.0, w0 * 20.0)

    ax_group.grid(True, which='both', linestyle=':', alpha=0.7)

    # --------------------------------------------------------
    # POLE-ZERO MAP
    # --------------------------------------------------------

    poles = np.array([p1, p2], dtype=complex)
    zeros = np.array([z1, z2], dtype=complex)

    ax_pz.scatter(np.real(poles), np.imag(poles), marker='x', s=90, linewidths=2.0, label='Poles')
    ax_pz.scatter(np.real(zeros), np.imag(zeros), marker='o', s=70, facecolors='none', linewidths=2.0, label='Zeros')

    ax_pz.axhline(0.0, color='black', linewidth=0.9)
    ax_pz.axvline(0.0, color='black', linewidth=0.9)

    limit = 1.25 * w0

    ax_pz.set_xlim(-limit, limit)
    ax_pz.set_ylim(-limit, limit)
    ax_pz.set_aspect('equal', adjustable='box')

    ax_pz.set_title('Pole-Zero Map', fontsize=10)
    ax_pz.set_xlabel(r'$\sigma$')
    ax_pz.set_ylabel(r'$j\omega$')

    ax_pz.grid(True, linestyle=':', alpha=0.7)
    ax_pz.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0)

    # --------------------------------------------------------
    # AXIS FORMAT
    # --------------------------------------------------------

    for ax in [ax_mag, ax_phase, ax_group]:

        ax.tick_params(axis='x', labelsize=8)
        ax.tick_params(axis='y', labelsize=8)

    ax_pz.tick_params(axis='x', labelsize=8)
    ax_pz.tick_params(axis='y', labelsize=8)

    fig.subplots_adjust(left=0.08, right=0.86, bottom=0.09, top=0.93, hspace=0.40, wspace=0.32)

    plt.show()
    plt.close(fig)

# ------------------------------------------------------------
# INTERACTIVE WIDGET
# ------------------------------------------------------------

widget_plot = interactive(plot_allpass_filter, w0=w0_slider, Q=q_slider)

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

plot_output = widget_plot.children[-1]

# ------------------------------------------------------------
# REMOVE OUTPUT SCROLL BARS
# ------------------------------------------------------------

display(HTML("""
<style>

.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output,
.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.output_scroll {
    max-height: none !important;
    height: auto !important;
    overflow: visible !important;
    overflow-y: visible !important;
    overflow-x: visible !important;
}

</style>
"""))

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

display(intro_html)
display(control_row)
display(parameter_box)
display(plot_output)